# Single enzyme — time-span exploration

Simulates one trajectory of the `SingleEnzyme` ODE with a few substrate boluses.
Vary `T_SPAN` in the config cell to find a time window where the dynamics are interesting
(not too fast to miss, not so long everything has equilibrated).

In [ ]:
import sys
from pathlib import Path

_here = Path().resolve()
_llo = next(
    (p for p in [_here, _here.parent, _here.parent.parent]
     if (p / 'create_dataset.py').exists()),
    _here,
)
if str(_llo) not in sys.path:
    sys.path.insert(0, str(_llo))

import numpy as np
import matplotlib.pyplot as plt
from sim.Single_enzyme import SingleEnzyme
from sim.syndata_simulator_ODE import simulate_ivp_with_bolus, single_event_generator

In [ ]:
# ── Config — edit here ────────────────────────────────────────────────────
T_SPAN = 10.0    # <-- vary this to explore timescales
N_STEPS = 400
SEED = 42
# ─────────────────────────────────────────────────────────────────────────

In [ ]:
# Sample one theta from the supervisor's ranges U(1, 200) per parameter
rng = np.random.default_rng(SEED)
param_ranges = [(1.0, 200.0)] * 6
theta = np.array([rng.uniform(lo, hi) for lo, hi in param_ranges], dtype=np.float32)

_, _, names_full = SingleEnzyme(None, None, None, dim=True)
param_names = ["kcat_f", "kcat_r", "Ka", "Kb", "Kc", "Kd"]
print("Theta:")
for name, val in zip(param_names, theta):
    print(f"  {name:8s} = {val:.3f}")

In [ ]:
# A few substrate boluses spread across the trajectory
n_boluses = 5
bolus_times = np.linspace(T_SPAN * 0.05, T_SPAN * 0.75, n_boluses)
events = []
for i, t in enumerate(bolus_times):
    channel = "A" if i % 2 == 0 else "B"
    events.append((float(t), channel, 2.0))

print("Bolus events (time, channel, amount):")
for e in events:
    print(f"  t={e[0]:.2f}  {e[1]}  +{e[2]}")

In [ ]:
# Initial conditions: A=1, B=1, C=0, D=0, E=1, I=0
x0 = np.array([1.0, 1.0, 0.0, 0.0, 1.0, 0.0], dtype=np.float32)

t_sol, x_sol = simulate_ivp_with_bolus(
    SingleEnzyme,
    k=theta,
    y0=x0,
    t_start=0.0,
    t_end=T_SPAN,
    bolus_gen=single_event_generator(events),
    species_names=list(names_full),
)
t_sol = np.asarray(t_sol)
x_sol = np.asarray(x_sol)
print(f"Solved: {x_sol.shape[0]} time points over t=[0, {T_SPAN}]")

In [ ]:
state_names = list(names_full)
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
bolus_colors = {"A": "steelblue", "B": "tomato"}

fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
axes_flat = axes.flatten()

for i, (ax, name) in enumerate(zip(axes_flat, state_names)):
    ax.plot(t_sol, x_sol[:, i], color=colors[i], linewidth=1.8)
    for t_b, ch, _ in events:
        ax.axvline(t_b, color=bolus_colors.get(ch, "grey"), alpha=0.4, linewidth=1.2, linestyle="--")
    ax.set_title(name, fontsize=11)
    ax.set_ylabel("concentration", fontsize=9)
    ax.grid(True, alpha=0.25)

for ax in axes[1]:
    ax.set_xlabel("time", fontsize=9)

import matplotlib.patches as mpatches
legend_handles = [
    mpatches.Patch(color=bolus_colors["A"], alpha=0.6, label="A bolus"),
    mpatches.Patch(color=bolus_colors["B"], alpha=0.6, label="B bolus"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f"SingleEnzyme trajectory  |  T_SPAN={T_SPAN}  |  seed={SEED}",
             fontsize=12)
fig.tight_layout()
plt.show()